# Storage

The Candidate Generators step built four candidate generators that all run in memory against a 5 000-story list.
That works fine in a notebook. It breaks at production scale.

This step introduces a storage layer that would sit behind a real production system:

```
JSONL file ──(one-time migration)──► Postgres  (source of truth)
                                         │
                               ┌─────────┼──────────┐
                               ▼         ▼          ▼
                         Elasticsearch  Qdrant   Neo4j
                         (BM25 index)  (ANN)    (graph)
```

**Postgres** is the write-ahead store — every story and user is written here first.
The other three are derived indexes that can be rebuilt at any time from Postgres.
Each one is purpose-built for its access pattern.

Every generator interface from the Candidate Generators step stays identical. Only the backend changes.

In [1]:
import html
import json
import sys
import time
from pathlib import Path

from IPython.display import HTML, display

_src = Path('../../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from ranking_service.interface import Context, Story, User
from ranking_service.impl_handrolled import HandrolledRankingService
from ranking_service.encoder import SentenceEncoder
from ranking_service.backends import (
    PostgresStore,
    ElasticsearchCandidateGenerator,
    QdrantCandidateGenerator,
    Neo4jCandidateGenerator,
)

/Users/dev/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/dev/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def show(stories, n=5, extra=None):
    extra = extra or {}
    cols = list(extra.keys()) + ['points', 'title', 'author', 'comments']
    header = '<tr>' + ''.join(
        f'<th style="padding:4px 10px;text-align:left">{c}</th>' for c in cols
    ) + '</tr>'
    rows = []
    for s in stories[:n]:
        cells = []
        for fn in extra.values():
            val = fn(s)
            cells.append(f'{val:.3f}' if isinstance(val, float) else str(val))
        cells += [
            str(s.points),
            html.escape((s.title or '')[:80]),
            html.escape(s.author or ''),
            str(s.num_comments),
        ]
        rows.append('<tr>' + ''.join(
            f'<td style="padding:4px 10px">{c}</td>' for c in cells
        ) + '</tr>')
    display(HTML(f'<table>{header}{chr(10).join(rows)}</table>'))

---
## Why memory breaks at scale

Candidate Generators step generators all loop over the story list in Python.
Let's see what that costs as the corpus grows.

In [3]:
import numpy as np

# Load snapshot and encoder just for the scale-math cell
snapshot_path = Path('../../../data').resolve() / 'hn_stories_snapshot.jsonl'
raw = [json.loads(l) for l in snapshot_path.read_text().splitlines() if l.strip()]
_stories_tmp = [
    Story(id=r['id'], title=r['title'], url=r.get('url'), points=r.get('points') or 0,
          author=r['author'], created_at=r['created_at'], num_comments=r.get('num_comments') or 0)
    for r in raw
]
cache_path = Path('../../../data').resolve() / 'story_embeddings.npy'
_enc_tmp = SentenceEncoder(cache_path=cache_path).fit(_stories_tmp)

alice_tmp = User(id='alice', interests=['machine learning', 'large language models',
                                        'transformers', 'neural networks'])
t0 = time.perf_counter()
_ = _enc_tmp.most_similar(alice_tmp.interests, top_k=500)
linear_5k_s = time.perf_counter() - t0

print(f'Linear cosine scan ({_enc_tmp.n_dims} dims):')
print(f'  {"Stories":>12}  {"Time":>14}')
for n_stories, t in [
    (5_000,     linear_5k_s),
    (100_000,   linear_5k_s * 20),
    (1_000_000, linear_5k_s * 200),
    (5_000_000, linear_5k_s * 1000),
]:
    flag = '  ← we are here' if n_stories == 5_000 else ''
    unit = f'{t*1000:.0f} ms' if t < 1 else f'{t:.1f} s' if t < 60 else f'{t/60:.1f} min'
    print(f'  {n_stories:>12,}  {unit:>14}{flag}')

print()
print('Substring scan and in-memory author lookup scale the same way.')
print('Each storage backend below solves this for its access pattern.')

Loaded cached embeddings (story_embeddings.npy)
Linear cosine scan (384 dims):
       Stories            Time
         5,000           45 ms  ← we are here
       100,000          900 ms
     1,000,000           9.0 s
     5,000,000          45.0 s

Substring scan and in-memory author lookup scale the same way.
Each storage backend below solves this for its access pattern.


---
## Docker and Docker Compose — ELI5

**Docker** packages an app and all its dependencies into a sealed image — like a shipping
container. The same image runs identically on your laptop, a teammate's machine, or a cloud server.

**Docker Compose** runs several containers together and wires them up. One command starts everything:

```bash
# from the repo root
docker compose -f infra/docker-compose.yml --profile core --profile storage-deep-dive up -d
```

- `--profile core` starts **Postgres** (the source of truth, used in every step)
- `--profile storage-deep-dive` starts **Elasticsearch**, **Qdrant**, and **Neo4j**

```
infra/docker-compose.yml
├── postgres       → localhost:5432   (source of truth — stories + users)
├── elasticsearch  → localhost:9200   (BM25 keyword search)
├── qdrant         → localhost:6333   (ANN vector search)
└── neo4j          → localhost:7687   (graph traversal)
```

In [4]:
# Health-check all four services
from elasticsearch import Elasticsearch
from qdrant_client import QdrantClient
from neo4j import GraphDatabase
import psycopg2

es_client     = Elasticsearch('http://localhost:9200', request_timeout=5)
qdrant_client = QdrantClient(host='localhost', port=6333, timeout=5, check_compatibility=False)
neo4j_driver  = GraphDatabase.driver('bolt://localhost:7687', auth=('neo4j', 'mlinfra123'))
pg_store      = PostgresStore.connect()

checks = {
    'Postgres':        lambda: pg_store.story_count() >= 0 and 'ok',
    'Elasticsearch':   lambda: es_client.info()['version']['number'],
    'Qdrant':          lambda: qdrant_client.get_collections() and 'ok',
    'Neo4j':           lambda: neo4j_driver.verify_connectivity() or 'ok',
}

all_up = True
for name, check in checks.items():
    try:
        result = check()
        print(f'  {name:<20} UP')
    except Exception as e:
        print(f'  {name:<20} DOWN  ({e})')
        all_up = False

if not all_up:
    print('\nRun: docker compose -f infra/docker-compose.yml --profile core --profile storage-deep-dive up -d')
else:
    print('\nAll services healthy — ready to ingest.')

  Postgres             UP
  Elasticsearch        UP
  Qdrant               UP


  Neo4j                UP

All services healthy — ready to ingest.


---
## Source of truth — Postgres

Before we index anything into ES, Qdrant, or Neo4j, we need a place that owns the data.
That's Postgres.

**Why a relational database as the source of truth?**
- **Transactions**: a story write either fully commits or fully rolls back. No partial writes.
- **Durability**: data survives restarts; the search indexes can be wiped and rebuilt from here.
- **Queries**: find stories by author, filter by date range, join with user engagement data — SQL handles all of it without a specialized index.

The `stories` table mirrors the `Story` dataclass exactly. The `users` table stores
user profiles and their interests as a Postgres array — a clean fit for a list of strings.

```sql
CREATE TABLE stories (
    id            TEXT PRIMARY KEY,
    title         TEXT NOT NULL,
    url           TEXT,
    points        INTEGER DEFAULT 0,
    author        TEXT,
    created_at    TEXT,
    num_comments  INTEGER DEFAULT 0,
    is_ask_hn     BOOLEAN DEFAULT FALSE,
    is_show_hn    BOOLEAN DEFAULT FALSE,
    is_front_page BOOLEAN DEFAULT FALSE
);

CREATE TABLE users (
    id        TEXT PRIMARY KEY,
    interests TEXT[] DEFAULT '{}',
    language  TEXT,
    country   TEXT
);
```

In [5]:
# ── INGESTION — run once, then comment out ──────────────────────────────────
pg_store.setup_schema(drop_existing=True)

# Migrate stories from the JSONL snapshot into Postgres
stories_from_file = [
    Story(
        id=r['id'], title=r['title'], url=r.get('url'),
        points=r.get('points') or 0, author=r['author'],
        created_at=r['created_at'], num_comments=r.get('num_comments') or 0,
        is_ask_hn=r.get('is_ask_hn', False),
        is_show_hn=r.get('is_show_hn', False),
        is_front_page=r.get('is_front_page', False),
    )
    for r in raw
]
pg_store.ingest_stories(stories_from_file)

# Save users — in production these come from signup/profile updates
users_to_save = [
    User(id='alice', interests=['machine learning', 'large language models',
                                'transformers', 'neural networks']),
    User(id='bob',   interests=['trading', 'markets', 'finance', 'economics']),
]
pg_store.ingest_users(users_to_save)
# ────────────────────────────────────────────────────────────────────────────

Upserted 5000 stories into Postgres
Upserted 2 users into Postgres


In [6]:
# From here on, stories and users come FROM Postgres — not from the file
stories = pg_store.load_stories()
users   = pg_store.load_users()

print(f'Loaded from Postgres:')
print(f'  {pg_store.story_count():>6} stories')
print(f'  {pg_store.user_count():>6} users')
print()
for u in users:
    print(f'  user={u.id:<8}  interests={u.interests}')
print()
print('Sample stories:')
show(stories, n=3)

Loaded from Postgres:
    5000 stories
       2 users

  user=alice     interests=['machine learning', 'large language models', 'transformers', 'neural networks']
  user=bob       interests=['trading', 'markets', 'finance', 'economics']

Sample stories:


points,title,author,comments
1,India's Unconvincing Economic Facade,eatonphil,0
1,Russian politician says GTA 6 carries 'the stench of Americanism',HelloUsername,0
1,"Mu: A social app with a backbone, built on European values",modinfo,0


In [7]:
# Load the encoder (uses cached .npy — no re-embedding needed)
from datetime import datetime, timezone

cache_path = Path('../../../data').resolve() / 'story_embeddings.npy'
encoder = SentenceEncoder(cache_path=cache_path).fit(stories)

_times = [datetime.fromisoformat(s.created_at.replace('Z', '+00:00')) for s in stories]
DATA_NOW = max(_times)

ctx   = Context(timestamp=DATA_NOW.isoformat())
alice = next(u for u in users if u.id == 'alice')
bob   = next(u for u in users if u.id == 'bob')

print(f'Encoder: {encoder.n_stories} stories x {encoder.n_dims} dims')
print(f'Alice interests (from Postgres): {alice.interests}')
print(f'Bob   interests (from Postgres): {bob.interests}')

Loaded cached embeddings (story_embeddings.npy)
Encoder: 5000 stories x 384 dims
Alice interests (from Postgres): ['machine learning', 'large language models', 'transformers', 'neural networks']
Bob   interests (from Postgres): ['trading', 'markets', 'finance', 'economics']


---
## Backend 1 — Elasticsearch

**Replaces**: `KeywordCandidateGenerator` (Python substring scan)

### Inverted index vs substring scan

Substring match checks `keyword in title` for every story — O(N) per query.

Elasticsearch pre-builds an **inverted index**: for each word, store the list of
documents that contain it. Looking up `"transformers"` at query time is an O(1)
dictionary lookup, not a scan.

On top of that, **BM25** scoring weights rare terms higher than common ones.
`"RLHF"` in a title is a strong signal; `"the"` is noise. Exact substring match
treats them equally.

```
Substring:  scan N titles → O(N)
ES index :  word → [doc_ids] → O(1)  +  BM25 scoring
```

In [8]:
# ── INGESTION (from Postgres) — run once, then comment out ──────────────────
ElasticsearchCandidateGenerator.ingest(stories, es_client, recreate=True)
# ────────────────────────────────────────────────────────────────────────────

Indexed 5000 stories into 'hn_stories'


In [9]:
es_gen   = ElasticsearchCandidateGenerator(es_client)
es_alice = es_gen.get_candidates(stories, alice)
es_bob   = es_gen.get_candidates(stories, bob)

print(f'Alice: {len(es_alice)} candidates  |  Bob: {len(es_bob)} candidates')
display(HTML('<h4>Elasticsearch — Alice</h4>'))
show(es_alice, n=5)
display(HTML('<h4>Elasticsearch — Bob</h4>'))
show(es_bob, n=5)

Alice: 272 candidates  |  Bob: 73 candidates


points,title,author,comments
6,Large Language Models Are Overkill. Enter the Small Language Model,zerogpu,1
122,Knowledge Distillation of Black-Box Large Language Models (2024),babelfish,23
3,Graph of Thoughts: Solving Elaborate Problems with Large Language Models,simonpure,0
1,Guide to Using Large Language Models and Generative AI in Economic History,paulpauper,0
2,Tapered Language Models,sonabinu,0


points,title,author,comments
2,Vibe-Trading: Your Personal Trading Agent,grajmanu,0
4,Agentic Trading on Robinhood,huragok,2
1,EU Trade Explorer,abracadabrapouf,1
3,"Mathematical finance, formally verified in Lean 4",raphaelrrcoelho,0
1,India's Unconvincing Economic Facade,eatonphil,0


In [10]:
# BM25 vs substring: what does BM25 catch that exact match misses?
from ranking_service.candidates import KeywordCandidateGenerator

kw_alice = KeywordCandidateGenerator().get_candidates(stories, alice)

es_ids = {s.id for s in es_alice}
kw_ids = {s.id for s in kw_alice}
bm25_only = [s for s in es_alice if s.id not in kw_ids]

print(f'Substring match : {len(kw_ids):>4} candidates')
print(f'BM25 (ES)       : {len(es_ids):>4} candidates')
print(f'Overlap         : {len(es_ids & kw_ids):>4}')
print(f'BM25 only       : {len(bm25_only):>4}  ← stories ES finds that exact substring misses')
if bm25_only:
    print()
    print('Examples BM25 catches via stemming / partial match:')
    for s in bm25_only[:5]:
        print(f'  "{s.title}"')

Substring match :    7 candidates
BM25 (ES)       :  272 candidates
Overlap         :    7
BM25 only       :  265  ← stories ES finds that exact substring misses

Examples BM25 catches via stemming / partial match:
  "Tapered Language Models"
  "Tokki – language learning app because Duolingo is useless"
  "Transformations"
  "Some learnings from temporal.io building SDKs for 8 languages"
  "Zero Weights Language Model (MSE-GLM)"


---
## Backend 2 — Qdrant

**Replaces**: `SemanticCandidateGenerator` (linear cosine scan)

### ANN vs exact cosine

Exact cosine computes the dot product of the query against every vector — O(N).
At 5M stories × 384 dims that's ~2 billion multiplications per search.

Qdrant builds an **HNSW** graph (Hierarchical Navigable Small World): a multi-layer
structure where each node connects to its nearest neighbours. Queries start at the
top layer (coarse) and drill down — visiting a small fraction of vectors to find
approximate neighbours.

The trade-off: ~1-2% recall loss for a 100-1000× speedup.
For candidate generation (where a ranker re-scores the pool anyway) this is always worth it.

The **same fastembed vectors** from the Candidate Generators step are used — the encoder is unchanged.

In [11]:
# ── INGESTION (from Postgres) — run once, then comment out ──────────────────
QdrantCandidateGenerator.ingest(stories, encoder, qdrant_client, recreate=True)
# ────────────────────────────────────────────────────────────────────────────

Uploaded 5000 vectors to Qdrant collection 'hn_stories'


In [12]:
qdrant_gen   = QdrantCandidateGenerator(qdrant_client, encoder)
qdrant_alice = qdrant_gen.get_candidates(stories, alice)
qdrant_bob   = qdrant_gen.get_candidates(stories, bob)

print(f'Alice: {len(qdrant_alice)} candidates  |  Bob: {len(qdrant_bob)} candidates')
display(HTML('<h4>Qdrant ANN — Alice</h4>'))
show(qdrant_alice, n=5)
display(HTML('<h4>Qdrant ANN — Bob</h4>'))
show(qdrant_bob, n=5)

Alice: 500 candidates  |  Bob: 500 candidates


points,title,author,comments
6,Large Language Models Are Overkill. Enter the Small Language Model,zerogpu,1
2,Tapered Language Models,sonabinu,0
122,Knowledge Distillation of Black-Box Large Language Models (2024),babelfish,23
3,Graph of Thoughts: Solving Elaborate Problems with Large Language Models,simonpure,0
223,"Apple Neural Engine: Architecture, Programming, and Performance",Jimmc414,27


points,title,author,comments
2,Vibe-Trading: Your Personal Trading Agent,grajmanu,0
1,EU Trade Explorer,abracadabrapouf,1
1,MarketNow,eddyflores,1
2,"Show HN: Apex Trading Signals – AI commodity trade ideas, free Android beta",dmaso191,0
4,Agentic Trading on Robinhood,huragok,2


In [13]:
# ANN recall: how closely does Qdrant match exact cosine on this corpus?
exact_top100 = [s for s, _ in encoder.most_similar(alice.interests, top_k=100)]
exact_ids    = {s.id for s in exact_top100}
qdrant_ids   = {s.id for s in qdrant_alice[:100]}
overlap = len(exact_ids & qdrant_ids)

print(f'Top-100 exact cosine vs Qdrant ANN:')
print(f'  Exact cosine : {len(exact_ids)}')
print(f'  Qdrant ANN   : {len(qdrant_ids)}')
print(f'  Overlap      : {overlap}/100  ({overlap}% recall)')
print()
print('~99% recall is typical for HNSW at these settings.')
print('The 1-2 boundary stories that differ are not meaningful for downstream ranking.')

Top-100 exact cosine vs Qdrant ANN:
  Exact cosine : 100
  Qdrant ANN   : 100
  Overlap      : 100/100  (100% recall)

~99% recall is typical for HNSW at these settings.
The 1-2 boundary stories that differ are not meaningful for downstream ranking.


---
## Backend 3 — Neo4j

**Replaces**: `ConnectedCandidateGenerator` (in-memory author lookup)

### Why a graph database?

The author-affinity logic from Candidate Generators step:
1. Find Alice's top semantic matches → extract their authors
2. Return other stories by those authors

Step 2 is a Python list scan. In SQL it's a JOIN that requires full index scans.
In Neo4j it's two hops on a graph — and traversal cost is proportional to
**neighbourhood size**, not total graph size.

```cypher
MATCH (seed:Story)<-[:WROTE]-(a:Author)-[:WROTE]->(other:Story)
WHERE seed.id IN $seed_ids
  AND NOT other.id IN $seed_ids
RETURN DISTINCT other
```

In production this graph also encodes click and comment edges — authors a user
has engaged with score higher. That's a collaborative filtering signal without
needing a full recommendation model.

In [14]:
# ── INGESTION (from Postgres) — run once, then comment out ──────────────────
Neo4jCandidateGenerator.ingest(stories, neo4j_driver)
# ────────────────────────────────────────────────────────────────────────────

Neo4j graph: 2921 Author nodes, 5000 Story nodes, WROTE edges connecting them


In [15]:
neo4j_gen = Neo4jCandidateGenerator(neo4j_driver, encoder, seed_top_k=20)

# Show the traversal step-by-step
top20 = encoder.most_similar(alice.interests, top_k=20)
seed_authors = list(dict.fromkeys(s.author for s, _ in top20 if s.author))

print('Cypher executed by get_candidates():')
print('''
    MATCH (seed:Story)<-[:WROTE]-(a:Author)-[:WROTE]->(other:Story)
    WHERE seed.id IN $seed_ids AND NOT other.id IN $seed_ids
    RETURN DISTINCT other LIMIT $limit
''')
print(f'Step 1 — {len(top20)} seed stories → {len(seed_authors)} unique authors:')
for author in seed_authors:
    print(f'  {author}')

Cypher executed by get_candidates():

    MATCH (seed:Story)<-[:WROTE]-(a:Author)-[:WROTE]->(other:Story)
    WHERE seed.id IN $seed_ids AND NOT other.id IN $seed_ids
    RETURN DISTINCT other LIMIT $limit

Step 1 — 20 seed stories → 20 unique authors:
  zerogpu
  sonabinu
  babelfish
  simonpure
  Jimmc414
  paulpauper
  fodokidza
  shallow-mind
  bharadwajp
  LorenDB
  leschak
  Enjoyooor
  grajmanu
  KnoxProtocol
  krussikas
  dboon
  DarenWatson
  droidjj
  zphou
  TroubleSprouter


In [16]:
neo4j_alice = neo4j_gen.get_candidates(stories, alice)
neo4j_bob   = neo4j_gen.get_candidates(stories, bob)

print(f'Step 2 — graph traversal:')
print(f'  Alice: {len(neo4j_alice)} connected candidates')
print(f'  Bob  : {len(neo4j_bob)} connected candidates')
print()
display(HTML('<h4>Neo4j — Alice connected candidates</h4>'))
show(neo4j_alice, n=5)
display(HTML('<h4>Neo4j — Bob connected candidates</h4>'))
show(neo4j_bob, n=5)

Step 2 — graph traversal:
  Alice: 95 connected candidates
  Bob  : 61 connected candidates



points,title,author,comments
3,InfiniteDiffusion: A new approach to infinite generation with diffusion models,LorenDB,1
5,Linux Foundation Launches Akrites to Defend FOSS from AI-Enabled Exploits,LorenDB,0
94,"One man, two kernels, and a lot of RISC-V",LorenDB,15
2,Auto-Charge Tracker makes Steam Controller move toward its charging dock,LorenDB,0
58,SpaceX plans to launch Starlink mobile service in the US,LorenDB,1


points,title,author,comments
3,AI agents are sensitive to nudges,paulpauper,1
2,New Business Formation Is Surging–Again,paulpauper,0
3,Architectural Studies #02,paulpauper,0
3,Why American data centers can't plug in,paulpauper,0
1,The Duck Is Growing,paulpauper,1


---
## Climax — same pipeline, production backends

Load users from Postgres. Swap the Candidate Generators step generator instances for Storage step ones.
Every other line is identical to the Candidate Generators step.

In [17]:
from ranking_service.candidates import PopularityCandidateGenerator, ExplorationCandidateGenerator
from ranking_service.impl_handrolled import _hn_score

ranker = HandrolledRankingService()

# Users loaded from Postgres — not hardcoded
for user in users:
    print(f'\n=== {user.id.upper()} (interests from Postgres: {user.interests}) ===')

    pop_candidates  = PopularityCandidateGenerator().get_candidates(stories, user)
    kw_candidates   = es_gen.get_candidates(stories, user)           # Elasticsearch
    sem_candidates  = qdrant_gen.get_candidates(stories, user)       # Qdrant
    conn_candidates = neo4j_gen.get_candidates(stories, user)        # Neo4j

    known_ids = {s.id for s in kw_candidates} | {s.id for s in sem_candidates}
    exp_candidates = ExplorationCandidateGenerator(points_threshold=5, seed=42).get_candidates(
        stories, user, n=50, exclude_ids=known_ids
    )

    seen, combined = set(), []
    for s in pop_candidates + kw_candidates + sem_candidates + conn_candidates + exp_candidates:
        if s.id not in seen:
            seen.add(s.id)
            combined.append(s)

    print(f'  Popularity    : {len(pop_candidates):>4}')
    print(f'  Elasticsearch : {len(kw_candidates):>4}  (BM25)')
    print(f'  Qdrant        : {len(sem_candidates):>4}  (ANN)')
    print(f'  Neo4j         : {len(conn_candidates):>4}  (graph)')
    print(f'  Exploration   : {len(exp_candidates):>4}  (long tail)')
    print(f'  Combined      : {len(combined):>4}  unique')

    ranked_combined = ranker.rank(combined,       user, ctx)
    ranked_sem_only = ranker.rank(sem_candidates, user, ctx)

    display(HTML(
        f'<h4>{user.id.capitalize()} — HN decay on full combined pool '
        f'({len(combined)} candidates, popularity-dominated)</h4>'
    ))
    show(ranked_combined, n=5, extra={'hn_score': lambda s: _hn_score(s, DATA_NOW)})

    display(HTML(
        f'<h4>{user.id.capitalize()} — HN decay on semantic pool only '
        f'({len(sem_candidates)} candidates, interest-filtered)</h4>'
    ))
    show(ranked_sem_only, n=5, extra={'hn_score': lambda s: _hn_score(s, DATA_NOW)})



=== ALICE (interests from Postgres: ['machine learning', 'large language models', 'transformers', 'neural networks']) ===


  Popularity    :  500
  Elasticsearch :  272  (BM25)
  Qdrant        :  500  (ANN)
  Neo4j         :   95  (graph)
  Exploration   :   50  (long tail)
  Combined      : 1243  unique


hn_score,points,title,author,comments
42.070,566,Claude Code Is Steganographically Marking Requests,kirushik,174
24.841,357,The labor share of income in the US is at its lowest post-war level,loughnane,337
12.996,150,County with 37 Data Centers Asks Schools to 'Conserve Electricity',01-_-,83
11.028,114,"EU commissioners shut down air conditioning for employees, leave theirs on",spwa4,92
10.839,609,European digital ID wallets rely on safety services of Google and Apple,donohoe,258


hn_score,points,title,author,comments
1.283,995,Age verification is just a precursor to automated attribution of speech,arkhiver,611
1.146,241,"LongCat-2.0, a large-scale MoE model with 1.6T total and 48B Active",benjiro29,67
0.919,191,Memory Safe Context Switching,modeless,29
0.674,250,Ornith-1.0: self-improving open-source models for agentic coding,danboarder,48
0.518,105,Exploring PDP-1 Lisp (1960),ozymandiax,23



=== BOB (interests from Postgres: ['trading', 'markets', 'finance', 'economics']) ===
  Popularity    :  500
  Elasticsearch :   73  (BM25)
  Qdrant        :  500  (ANN)
  Neo4j         :   61  (graph)
  Exploration   :   50  (long tail)
  Combined      : 1093  unique


hn_score,points,title,author,comments
42.070,566,Claude Code Is Steganographically Marking Requests,kirushik,174
24.841,357,The labor share of income in the US is at its lowest post-war level,loughnane,337
12.996,150,County with 37 Data Centers Asks Schools to 'Conserve Electricity',01-_-,83
11.028,114,"EU commissioners shut down air conditioning for employees, leave theirs on",spwa4,92
10.839,609,European digital ID wallets rely on safety services of Google and Apple,donohoe,258


hn_score,points,title,author,comments
5.705,93,We moved our Bluesky data to Eurosky,dotcoma,77
0.304,806,OpenRA,tosh,165
0.279,6,Show HN: Scaffold a BigQuery and dbt and Cube project an AI agent can operate,zubairov,3
0.228,630,Fintech Engineering Handbook,signa11,217
0.220,4,"We encode time in space, and pay in complexity",sxx0,1


**Finding**: the combined pool's top-5 is identical for Alice and Bob — the 500 popularity candidates dominate, and HN decay ranks by points and age, not by user interests. High-scoring recent stories win regardless of who's asking.

The semantic-only pool tells a different story: the same HN decay formula, applied to a pool already filtered by each user's interests, surfaces genuinely different content. Alice sees ML/LLM stories; Bob sees trading and economics.

This is the key insight: **candidate generation controls who gets *considered*. The ranker decides the order within that pool.** When the pool is interest-filtered, even a user-agnostic ranker produces personalised results.

The remaining gap: the ranker still can't actively boost a story *because* it matches Alice's interests — it only benefits from the filtered pool. The Ranker step fixes that with a FastAPI serving layer and Redis cache, and Part 2 introduces a learned ranker that reads the user signal directly.